# 07 · Test-mix control: why PR-AUC falls when fraud gets rarer

Question from the mid-evaluation: *the context gets bigger as the fraud rate goes down, so why does PR-AUC still fall?*

PR-AUC can fall for two different reasons, and the headline table (notebook 04) mixes them:

1. **The model learned less.** Its context held fewer frauds (about 200 at 20% with 1K rows, about 16 at 0.172% with 10K rows).
2. **The test set got harder to score well on.** Every test set holds the same 98 frauds, but next to them sit 392 legitimate rows at 20% and about 56,900 at 0.172%. More legitimate rows means more false alarms for the same model, so precision and PR-AUC drop even if the model ranks just as well.

This notebook holds reason 1 fixed. It takes the models built at the natural rate (0.172%) and scores them again on the test sets of the other fraud rates. No model is run again: the fraud-rate levels are nested, so the 20% test set is the natural test set with legitimate rows removed, and every row in it already has a score.

| Output | What it is |
| --- | --- |
| `rescore_by_test_mix_runs.csv` | one row per model × seed × test fraud rate |
| `rescore_by_test_mix_summary.csv` | mean over seeds, next to the headline PR-AUC of the model built at that rate |
| `figures/fig6_test_mix.png` | the two effects in one figure |

**CPU runtime** is enough. Needs the saved runs on Drive from notebooks 02 and 03.

In [ ]:
# 1. Drive, latest code, libraries (same as notebook 04).
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 2. Labels and the saved splits. The fingerprint check makes sure these are the
#    exact splits every earlier notebook used.
from fraudicl.config import DRIVE_DIR
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
train_rows = {s.level: len(s.train) for s in splits if s.seed == 0}
OUT = f"{DRIVE_DIR}/results"

# How big each test set is, for seed 0: the frauds stay at 98, only the
# legitimate rows change.
for s in splits:
    if s.seed == 0:
        print(f"{s.level:>8}: {len(s.test):>6} test rows, {int(y[s.test].sum())} frauds")

In [ ]:
# 3. Saved runs and the headline table (PR-AUC of the model built at each rate).
from fraudicl import report as R
runs = R.load_runs(OUT)
headline = R.headline_table(runs, train_rows)

In [ ]:
# 4. The control. For each natural-rate model (XGBoost on all rows, both foundation
#    models and XGBoost on the same 10K rows) and each seed: take its saved test
#    scores, keep only the rows that belong to the 20% / 5% / 1% / 0.5% test set,
#    and compute the metrics again. Scores and the F1 threshold (picked on the
#    natural validation set) are never changed.
control = R.rescore_by_test_mix(OUT, y, splits, train_rows)
print(control.groupby("model").seed.nunique())   # expect 5 seeds per model

# Check: on its own (natural) test set the control must give back the PR-AUC
# saved by notebook 02/03.
nat = control[control.test_level == "natural"]
saved = runs[runs.level.astype(str) == "natural"]
for model in nat.model.unique():
    a = nat[nat.model == model].sort_values("seed").pr_auc.values
    b = saved[(saved.model == model) & ((saved.k == 10_000) | (model == R.HEADLINE_XGB))
              & (saved.policy.isin(["uniform", "all rows"]))].sort_values("seed").pr_auc.values
    assert len(a) == len(b) and abs(a - b).max() < 1e-9, model
print("control reproduces the saved natural-rate PR-AUC")

In [ ]:
# 5. Mean over seeds. pr_auc_natural_model: the natural-rate model on each test mix.
#    pr_auc_own_model: the headline number, a model built at that rate.
summary = R.rescore_summary(control, headline)
cols = ["model", "test_level", "test_rows", "test_frauds", "pr_auc_natural_model",
        "pr_auc_own_model", "recall_at_fpr_0_1pct", "recall_at_val_threshold",
        "precision_at_val_threshold", "f1_at_val_threshold"]
summary[cols].round(3)

**How to read the table.** Take one model and read down `pr_auc_natural_model`: the model and its scores are the same in every row, so any change comes from the test set alone (reason 2). `recall_at_val_threshold` should be identical in every row (same 98 frauds, same scores, same threshold), while `precision_at_val_threshold` climbs as legitimate rows are removed: that is the mechanism. `recall_at_fpr_0_1pct` barely moves, because it only looks at the share of legitimate rows wrongly flagged, not their number.

Then compare `pr_auc_natural_model` with `pr_auc_own_model` at 20%: the gap is what a model built at 20% (more frauds in its context or training set) gained beyond the natural one (reason 1).

In [ ]:
# 6. Figure 6 and export. Download the zip and unzip it into results/ on the laptop.
import shutil
from IPython.display import Image, display
EXPORT = "/content/export_07"
os.makedirs(f"{EXPORT}/figures", exist_ok=True)
control.to_csv(f"{EXPORT}/rescore_by_test_mix_runs.csv", index=False)
summary.to_csv(f"{EXPORT}/rescore_by_test_mix_summary.csv", index=False)
R.plot_rescore_by_test_mix(summary, f"{EXPORT}/figures/fig6_test_mix.png")
display(Image(f"{EXPORT}/figures/fig6_test_mix.png"))
shutil.make_archive("/content/results_07", "zip", EXPORT)
from google.colab import files
files.download("/content/results_07.zip")